In [144]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


In [145]:
#DATA_PATH = "/home/denklewer/Data/gbm122/pVACseq/"
DATA_PATH = "D:/Google Drive/Work/ITMO/data/schreiber_full/"

In [146]:
from mhcgnomes import parse

# read total data

source_total_df =  pd.read_csv("{}schreiber_total_data_full.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(source_total_df['allele']))
per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt = pd.read_csv("{}schreiber_data_full_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    df_wt = pd.read_csv("{}schreiber_data_full_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";") 
    
    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt
    
    assert 'peptide' in df_mt.columns
    assert 'length' in df_mt.columns
    assert RESULT_COLUMNS['id'] in df_mt.columns
    assert 'peptide' in df_wt.columns
    assert 'length' in df_wt.columns
    assert RESULT_COLUMNS['id'] in df_wt.columns
    
    
def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None


def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None
ALLELES

['H-2-Dd', 'H-2-Ld', 'H-2-IEd', 'H-2-Kd', 'H-2-IAd']

# 1. Read all the predictions

In [147]:
import os


In [150]:
per_method_columns = {
    'NetMHC': ['netMHC-3.4_score', 'netMHC-3.4_affinity', 'netMHC-4.0_score',  'netMHC-4.0_affinity','netMHC-4.0_percentile_rank', \
               'netMHCpan-4.1_score_ba', 'netMHCpan-4.1_score_el', 'netMHCpan-4.1_affinity', 'netMHCpan-4.1_percentile_rank_ba', 'netMHCpan-4.1_percentile_rank_el',  
                'netMHCIIpan-4.0_score_ba', 'netMHCIIpan-4.0_score_el', 'netMHCIIpan-4.0_affinity', 'netMHCIIpan-4.0_percentile_rank_ba', 'netMHCIIpan-4.0_percentile_rank_el',],
    'netMHC-3.4': [ 'score', 'affinity'],
    'netMHC-4.0':[ 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':[ 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0' :['score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],

    'mhcflurry':['affinity', 'percentile_rank_ba', 'processing_score', 'score_el','percentile_rank_el' ],
    'mhcnuggets':[ 'score', 'affinity', 'percentile_rank'],
    'acme':['score'],
    'puffin':['score', 'epistemic_var', 'aleatoric_var', 'binding_likelihood', 'percentile_rank'],
    'mhcseqnet':['score'],
    'mhcattnnet':[ 'score'],
    'ghmm': ['HMMB', 'percentile_rank']
}



In [151]:
PREDICTION_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/schreiber_full/"
methods =  next(os.walk(PREDICTION_PATH))[1] # read all the folders in a prediction path
methods.remove('NetMHC')
methods.insert(0, 'NetMHC')

#['NetMHC', 'puffin', 'mhcnuggets', 'acme', 'mhcseqnet', 'mhcattnnet']

per_allele_per_method_predictions = dict()

def less_is_good(column):
    if column.find('affinity') >= 0 :
        return True
    if column.find('percentile_rank') >=0:
        return True
    if column in ['epistemic_var', 'aleatoric_var', 'HMMB']:
        return True
    return False

for allele_name in ALLELES:
    print(f"{allele_name}:", end=" ")
    per_allele_per_method_predictions[allele_name] = dict()
    for method in methods:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        prediction_df = pd.read_csv(f"{PREDICTION_PATH}{method}/{method}_predictions_allele_{target_allele_name}.csv", sep=';')

        #add diff 

        print(f"{method}", end=" ")
        for score_column in per_method_columns[method]:
            result_score_column = f"{method}_{score_column}" if not (method == 'NetMHC') else score_column
            wt_scores = prediction_df[result_score_column + '_wt'].values
            mt_scores = prediction_df[result_score_column + '_mt'].values
            wt_peptide =  prediction_df['peptide_wt'].values
            # for these columns mt shoud be less than wt. In this case wt/mt will be > 1 and this is good. 
            if less_is_good(score_column):
                advantage = np.array([wt_sc/(mt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
            # for these columns mt shoud be greater than wt. In this case mt/wt will be > 1 and this is good. 
            else:
                advantage = np.array([mt_sc/(wt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
            prediction_df[f"{result_score_column}_advantage"] = advantage



        per_allele_per_method_predictions[allele_name][method] = prediction_df
    print()
            

H-2-Dd: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-Ld: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-IEd: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-Kd: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-IAd: NetMHC ghmm mhcflurry mhcnuggets puffin 


In [152]:
per_allele_per_method_predictions[ALLELES[1]]['NetMHC']

,source_sequence_name,allele,peptide_mt,length,netMHC-3.4_score_mt,netMHC-3.4_affinity_mt,netMHC-3.4_percentile_rank_mt,netMHC-4.0_score_mt,netMHC-4.0_affinity_mt,netMHC-4.0_percentile_rank_mt,...,netMHCpan-4.1_score_ba_advantage,netMHCpan-4.1_score_el_advantage,netMHCpan-4.1_affinity_advantage,netMHCpan-4.1_percentile_rank_ba_advantage,netMHCpan-4.1_percentile_rank_el_advantage,netMHCIIpan-4.0_score_ba_advantage,netMHCIIpan-4.0_score_el_advantage,netMHCIIpan-4.0_affinity_advantage,netMHCIIpan-4.0_percentile_rank_ba_advantage,netMHCIIpan-4.0_percentile_rank_el_advantage
0,seq1,H-2-Ld,ALSGVSVYF,9,0.037,33498.0,NaN,0.041,32036.15,12.0,...,0.926477,0.553739,0.900962,0.843995,0.615784,NaN,NaN,NaN,NaN,NaN
1,seq2,H-2-Ld,QPAAAPAAAL,10,0.338,1289.0,NaN,0.149,9991.85,2.0,...,1.100174,0.958146,1.208604,1.226221,0.956021,NaN,NaN,NaN,NaN,NaN
2,seq3,H-2-Ld,LSGVSVYF,8,0.070,23375.0,NaN,0.053,28269.03,9.0,...,0.975785,0.473203,0.982838,0.957528,0.617716,NaN,NaN,NaN,NaN,NaN
3,seq4,H-2-Ld,QVLGGVNL,8,0.047,30194.0,NaN,0.084,20206.55,5.0,...,0.921650,0.843375,0.956670,0.890329,0.893870,NaN,NaN,NaN,NaN,NaN
4,seq5,H-2-Ld,AAAPAAAL,8,0.038,33038.0,NaN,0.014,42995.14,37.0,...,1.004657,1.990724,1.003599,1.008941,1.740253,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8733,seq17637,H-2-Ld,REPAPEWPSIN,11,0.006,46727.0,NaN,0.110,15260.97,3.5,...,0.952766,1.305873,0.964163,0.915400,1.180705,NaN,NaN,NaN,NaN,NaN
8734,seq17638,H-2-Ld,DKGKIEDTGGS,11,0.005,47178.0,NaN,0.004,47994.24,95.0,...,0.987747,0.999900,0.999556,0.999651,1.000000,NaN,NaN,NaN,NaN,NaN
8735,seq17639,H-2-Ld,NDFSVLPLPN,10,0.006,46677.0,NaN,0.004,47756.99,95.0,...,1.123433,1.999988,1.031495,1.121004,1.162494,NaN,NaN,NaN,NaN,NaN
8736,seq17640,H-2-Ld,EPAPEWPSINL,11,0.368,928.0,NaN,0.291,2139.38,0.6,...,1.007832,1.707622,1.011054,1.017058,1.588421,NaN,NaN,NaN,NaN,NaN


# 2. Merge predictions

In [153]:
per_allele_predictions = dict()
for allele_name in ALLELES:
    list_for_allele = list(per_allele_per_method_predictions[allele_name].values())

    result_df = list_for_allele[0].copy()
    for df_to_merge in list_for_allele[1:]:
        result_df = result_df.merge(df_to_merge, on=['source_sequence_name', 'peptide_mt', 'length', 'peptide_wt', 'allele'], how='left')
    result_df = result_df[['source_sequence_name','allele', 'length'] + [column for column in result_df.columns if column.endswith('mt')] 
                          + [column for column in result_df.columns if column.endswith('_advantage')] 
                          + [column for column in result_df.columns if column.endswith('wt')]] 
    per_allele_predictions[allele_name] = result_df


In [154]:
all_methods = list(set([item.split('_')[0] for item in result_df.columns if  item.split('_')[0] not in ['allele', 'peptide', 'length', 'source']]))
all_methods

['netMHC-3.4',
 'ghmm',
 'mhcflurry',
 'puffin',
 'netMHC-4.0',
 'mhcnuggets',
 'netMHCpan-4.1',
 'netMHCIIpan-4.0']

In [155]:
unsupported_alleles = {method : set() for method in all_methods}

def add_unsupported_allele(method, alleles):
    methods_set = set(all_methods)
    if (method in methods_set):
        unsupported_alleles[method].update(alleles)

for allele_name in ALLELES:
    if defineClass(allele_name) == 'II':
        add_unsupported_allele('netMHCpan-4.1', [allele_name])
        add_unsupported_allele('netMHC-3.4', [allele_name])
        add_unsupported_allele('netMHC-4.0', [allele_name])
        add_unsupported_allele('acme', [allele_name])
        add_unsupported_allele('mhcflurry', [allele_name])
    else:
        add_unsupported_allele('netMHCIIpan-4.0', [allele_name])
        
    if not defineOrganism(allele_name) == 'Human':
        add_unsupported_allele('mhcseqnet', [allele_name])
        add_unsupported_allele('mhcattnnet', [allele_name])
        add_unsupported_allele('acme', [allele_name])

#special cases
add_unsupported_allele('acme',["HLA-C*04:01", "HLA-C*03:04"])
add_unsupported_allele('puffin',["HLA-C*04:01"])



def drop_unsupported_columns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        for method in all_methods:
            if allele_name in unsupported_alleles[method]:
                columns_to_drop = [col.strip() for col in df.columns if col.startswith(method)] + ['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt']
                df = df.drop(columns=columns_to_drop, errors='ignore')    
            per_allele_df[allele_name] = df
        
            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return per_allele_df




# Add source info

In [156]:

source_total_columns = ['VAR_ID', 'mutant_id', 'transcript_id', 'peptide_offset',
                        'A20_RGT', 'A20_RDP', 'A20_RAD_REF',
                        'A20_RAD_ALT', 'A20_RAF_ALT', 'A20_RPKM_mut',  'source_sequence_name', 
                        'peptide_wt', 'peptide_mt' , 'allele',  'length'
                       ]
source_total_df = source_total_df[source_total_columns]
source_total_df

,VAR_ID,mutant_id,transcript_id,peptide_offset,A20_RGT,A20_RDP,A20_RAD_REF,A20_RAD_ALT,A20_RAF_ALT,A20_RPKM_mut,source_sequence_name,peptide_wt,peptide_mt,allele,length
0,4_147755805_T/A,Zfp984-K196N,ENSMUST00000105734,188,T/A,249,95,154,0.618474,5.234282,seq0,KYLTQREKL,KYLTQRENL,H-2-Kd,9
1,18_53706875_A/C,Cep120-L771R,ENSMUST00000049811,765,A/C,259,110,149,0.575290,3.492483,seq6,VYQVELERL,VYQVERERL,H-2-Kd,9
2,11_69804901_G/C,Tmem102-P82A,ENSMUST00000051025,79,G/C,90,54,36,0.400000,1.645296,seq11,RFPPQAELL,RFAPQAELL,H-2-Kd,9
3,3_104800372_T/C,Mov10-D635G,ENSMUST00000166979,627,T/C,181,82,99,0.546961,1.347878,seq14,APSNSGADL,APSNSGAGL,H-2-Kd,9
4,11_60031753_G/A,Pemt-A55V,ENSMUST00000102693,50,G/A,21,15,6,0.285714,0.061122,seq15,SFVAAVITI,SFVAVVITI,H-2-Kd,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130391,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,233,A/G,939,0,939,1.000000,10.112060,seq92523,MCNSSCMGGMNR,MCNGSCMGGMNR,H-2-IAd,12
130392,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,232,A/G,939,0,939,1.000000,10.112060,seq92527,YMCNSSCMGGMNR,YMCNGSCMGGMNR,H-2-IAd,13
130393,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,231,A/G,939,0,939,1.000000,10.112060,seq92528,KYMCNSSCMGGMN,KYMCNGSCMGGMN,H-2-IAd,13
130394,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,231,A/G,939,0,939,1.000000,10.112060,seq92534,KYMCNSSCMGGM,KYMCNGSCMGGM,H-2-IAd,12


In [157]:
def reorder_columns(df, source_total_columns, source_id_columns):
    general_info = ['mutant_id', RESULT_COLUMNS['id'], 'allele', 'length']
    
    source_columns = [source_column for source_column in source_total_columns 
                      if source_column not in general_info
                      and source_column not in source_id_columns
                      and source_column not in ['peptide_wt', 'peptide_mt']]


    percentile_columns_mt = [column for column in df.columns if column.find('percentile')>=0 and column.endswith('mt')]
    percentile_columns_wt = [column for column in df.columns if column.find('percentile')>=0 and column.endswith('wt')]
    percentile_columns = percentile_columns_mt + percentile_columns_wt
    not_percentile_columns = [column for column in df.columns 
                              if column not in percentile_columns 
                              and  column not in general_info 
                              and column not in ['peptide_wt', 'peptide_mt'] 
                              and column not in source_columns]

    columns_ordered = list()
    columns_ordered.extend(general_info)
    columns_ordered.append('peptide_mt')
    columns_ordered.extend(percentile_columns_mt)

    columns_ordered.append('peptide_wt')
    columns_ordered.extend(percentile_columns_wt)
    columns_ordered.extend(source_columns)
    columns_ordered.extend(not_percentile_columns)

            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return df[columns_ordered]


per_allele_predictions = drop_unsupported_columns(per_allele_predictions)



class_I_dfs = list()
class_II_dfs = list()
for allele_name in ALLELES:
    df_allele_source = source_total_df.loc[source_total_df['allele'] == allele_name].copy()
    df_predictions = per_allele_predictions[allele_name]
    df_result = df_predictions.merge(df_allele_source, on=['source_sequence_name', 'peptide_mt', 'length', 'peptide_wt', 'allele'])
    df_result = reorder_columns(df_result, source_total_columns, source_id_columns=['mutand_id'])
    per_allele_predictions[allele_name] = df_result
    if defineClass(allele_name) == 'I':
        class_I_dfs.append(df_result)
    elif defineClass(allele_name) == 'II':
        class_II_dfs.append(df_result)

class_I_total_df = pd.concat(class_I_dfs)
class_II_total_df = pd.concat(class_II_dfs)
    
    
    

In [158]:
class_I_total_df

,mutant_id,source_sequence_name,allele,length,peptide_mt,netMHC-4.0_percentile_rank_mt,netMHCpan-4.1_percentile_rank_ba_mt,netMHCpan-4.1_percentile_rank_el_mt,ghmm_percentile_rank_mt,mhcflurry_percentile_rank_ba_mt,...,ghmm_HMMB_wt,mhcflurry_affinity_wt,mhcflurry_processing_score_wt,mhcflurry_score_el_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,puffin_score_wt,puffin_epistemic_var_wt,puffin_aleatoric_var_wt,puffin_binding_likelihood_wt
0,Serpinf1-I204S,seq124,H-2-Dd,8,SALSSLLL,36.0,4.341,1.884,50.00025,10.15175,...,NaN,13139.408338,0.168184,0.013483,0.086906,19525.50,0.119096,0.000718,0.018987,3.899331e-02
1,Fndc7-S519F,seq194,H-2-Dd,9,FVTAVAETQ,46.0,82.973,53.462,50.00025,54.18775,...,NaN,24284.059134,0.024910,0.004513,0.029079,36502.88,0.023569,0.000114,0.001563,2.630989e-07
2,Fndc7-S519F,seq329,H-2-Dd,8,FVTAVAET,75.0,83.830,60.833,50.00025,85.00575,...,NaN,29054.427592,0.020173,0.003731,0.028266,36825.66,0.020565,0.000108,0.001339,2.134040e-08
3,Kntc1-D910A,seq401,H-2-Dd,8,GGACLLLL,0.7,3.263,8.293,50.00025,15.98025,...,NaN,16777.426978,0.022058,0.006387,0.186651,6635.94,0.207015,0.006835,0.037774,1.218492e-01
4,Ruvbl2-R253C,seq431,H-2-Dd,8,DVINSCTQ,60.0,95.208,53.077,50.00025,75.72375,...,NaN,28488.109146,0.173762,0.006531,0.037985,33149.65,0.016420,0.000055,0.000985,2.431140e-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8733,Sephs1-A176P,seq14195,H-2-Kd,8,PTTVCQPN,37.0,92.530,99.167,50.00025,93.94425,...,NaN,31587.679473,0.001968,0.003227,0.036197,33797.39,0.071430,0.000381,0.009629,8.161671e-03
8734,Gm1141-L342R,seq14197,H-2-Kd,8,PRLHHSSR,65.0,90.631,78.500,50.00025,77.64425,...,NaN,29461.855873,0.005839,0.003500,0.080422,20944.43,0.058461,0.000361,0.007093,3.540867e-03
8735,Gm1141-L342R,seq14200,H-2-Kd,11,PPRLHHSSRSC,60.0,85.753,70.833,50.00025,70.53175,...,NaN,28689.863950,0.001152,0.003532,0.057496,26841.04,0.095455,0.000500,0.015510,2.651579e-02
8736,Ssbp3-G353A,seq14201,H-2-Kd,8,PNNISAIS,75.0,92.988,88.333,50.00025,93.25975,...,NaN,31438.486048,0.001649,0.003239,0.034705,34347.13,0.079884,0.000562,0.011324,1.167089e-02


In [159]:
sum([len(df) for df in per_allele_mt_df.values()])

130396

In [160]:
len(class_I_total_df) + len(class_II_total_df)

130396

In [161]:
len(source_total_df)

130396

## Save total pred to file

In [162]:
# for decoy_key in DECOY_PREFIXES:
for allele_name in ALLELES:
    total_df = per_allele_predictions[allele_name]
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}total_predictions_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";", index=False)

In [163]:
class_I_total_df.to_csv("{}total_predictions_class_I.csv".format(PREDICTION_PATH),sep=";", index=False)
class_II_total_df.to_csv("{}total_predictions_class_II.csv".format(PREDICTION_PATH),sep=";", index=False)

In [164]:
class_I_total_df

,mutant_id,source_sequence_name,allele,length,peptide_mt,netMHC-4.0_percentile_rank_mt,netMHCpan-4.1_percentile_rank_ba_mt,netMHCpan-4.1_percentile_rank_el_mt,ghmm_percentile_rank_mt,mhcflurry_percentile_rank_ba_mt,...,ghmm_HMMB_wt,mhcflurry_affinity_wt,mhcflurry_processing_score_wt,mhcflurry_score_el_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,puffin_score_wt,puffin_epistemic_var_wt,puffin_aleatoric_var_wt,puffin_binding_likelihood_wt
0,Serpinf1-I204S,seq124,H-2-Dd,8,SALSSLLL,36.0,4.341,1.884,50.00025,10.15175,...,NaN,13139.408338,0.168184,0.013483,0.086906,19525.50,0.119096,0.000718,0.018987,3.899331e-02
1,Fndc7-S519F,seq194,H-2-Dd,9,FVTAVAETQ,46.0,82.973,53.462,50.00025,54.18775,...,NaN,24284.059134,0.024910,0.004513,0.029079,36502.88,0.023569,0.000114,0.001563,2.630989e-07
2,Fndc7-S519F,seq329,H-2-Dd,8,FVTAVAET,75.0,83.830,60.833,50.00025,85.00575,...,NaN,29054.427592,0.020173,0.003731,0.028266,36825.66,0.020565,0.000108,0.001339,2.134040e-08
3,Kntc1-D910A,seq401,H-2-Dd,8,GGACLLLL,0.7,3.263,8.293,50.00025,15.98025,...,NaN,16777.426978,0.022058,0.006387,0.186651,6635.94,0.207015,0.006835,0.037774,1.218492e-01
4,Ruvbl2-R253C,seq431,H-2-Dd,8,DVINSCTQ,60.0,95.208,53.077,50.00025,75.72375,...,NaN,28488.109146,0.173762,0.006531,0.037985,33149.65,0.016420,0.000055,0.000985,2.431140e-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8733,Sephs1-A176P,seq14195,H-2-Kd,8,PTTVCQPN,37.0,92.530,99.167,50.00025,93.94425,...,NaN,31587.679473,0.001968,0.003227,0.036197,33797.39,0.071430,0.000381,0.009629,8.161671e-03
8734,Gm1141-L342R,seq14197,H-2-Kd,8,PRLHHSSR,65.0,90.631,78.500,50.00025,77.64425,...,NaN,29461.855873,0.005839,0.003500,0.080422,20944.43,0.058461,0.000361,0.007093,3.540867e-03
8735,Gm1141-L342R,seq14200,H-2-Kd,11,PPRLHHSSRSC,60.0,85.753,70.833,50.00025,70.53175,...,NaN,28689.863950,0.001152,0.003532,0.057496,26841.04,0.095455,0.000500,0.015510,2.651579e-02
8736,Ssbp3-G353A,seq14201,H-2-Kd,8,PNNISAIS,75.0,92.988,88.333,50.00025,93.25975,...,NaN,31438.486048,0.001649,0.003239,0.034705,34347.13,0.079884,0.000562,0.011324,1.167089e-02


In [165]:
per_allele_predictions[allele_name]

,mutant_id,source_sequence_name,allele,length,peptide_mt,netMHCIIpan-4.0_percentile_rank_ba_mt,netMHCIIpan-4.0_percentile_rank_el_mt,ghmm_percentile_rank_mt,mhcnuggets_percentile_rank_mt,puffin_percentile_rank_mt,...,netMHCIIpan-4.0_score_ba_wt,netMHCIIpan-4.0_score_el_wt,netMHCIIpan-4.0_affinity_wt,ghmm_HMMB_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,puffin_score_wt,puffin_epistemic_var_wt,puffin_aleatoric_var_wt,puffin_binding_likelihood_wt
0,Apoe-V184L,seq11,H-2-IAd,13,EGAERGLSAIRER,34.67,0.11,0.07275,17.15725,26.71825,...,0.263079,0.827304,2902.48,25.401205,0.252915,3239.89,0.198457,0.007626,0.025252,0.076421
1,Ybx1-P11A,seq14,H-2-IAd,14,QPAAAPAAALSAAD,8.28,0.25,0.03675,0.29125,1.61325,...,0.386595,0.769551,762.73,25.174292,0.528704,163.91,0.431860,0.008535,0.029878,0.514388
2,Apoe-V184L,seq16,H-2-IAd,14,EGAERGLSAIRERL,12.66,0.11,0.06775,6.24925,19.16575,...,0.361486,0.841370,1000.82,25.368416,0.384247,782.36,0.236598,0.006031,0.027805,0.128482
3,Ybx1-P11A,seq17,H-2-IAd,13,QPAAAPAAALSAA,21.28,0.72,0.04275,2.10675,1.83925,...,0.306723,0.643526,1810.04,25.691868,0.373493,878.90,0.418411,0.009842,0.030926,0.483638
4,Apoe-V184L,seq18,H-2-IAd,14,REGAERGLSAIRER,17.13,0.11,0.21475,13.71450,16.55125,...,0.328468,0.832941,1430.57,26.299534,0.275817,2528.79,0.247490,0.007101,0.028147,0.144169
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52086,Trp53-S237G,seq92523,H-2-IAd,12,MCNGSCMGGMNR,95.00,95.00,98.15525,68.77975,89.94225,...,0.079869,0.000076,21070.17,39.372486,0.108507,15456.05,0.041586,0.000925,0.007130,0.000003
52087,Trp53-S237G,seq92527,H-2-IAd,13,YMCNGSCMGGMNR,89.28,95.00,98.79675,39.12325,69.60625,...,0.117195,0.000647,14069.36,39.691213,0.167082,8200.85,0.109384,0.006971,0.016590,0.007039
52088,Trp53-S237G,seq92528,H-2-IAd,13,KYMCNGSCMGGMN,88.09,95.00,99.52275,65.06575,48.65425,...,0.126469,0.001479,12726.12,39.184596,0.069825,23488.93,0.158280,0.012218,0.021797,0.035084
52089,Trp53-S237G,seq92534,H-2-IAd,12,KYMCNGSCMGGM,91.67,95.00,99.85925,48.36475,66.99325,...,0.111515,0.000686,14961.13,39.962365,0.116052,14244.40,0.107662,0.007560,0.016833,0.007128
